# 14_01_成功条件と対応のある比較を計算する

コード：`ch14/blocks/14_01_成功条件と対応のある比較を計算する.ipynb`

本文のコードをコードセルで直接実行します。

上からセルを実行してください。ローカルではコード一式と同じPython環境をJupyterのカーネルに選びます。Colabでコード一式が必要な場合は、準備セルで配布ZIPをアップロードします。実行確認の範囲は `VALIDATION_COLAB_20261110.md` を参照してください。


In [ ]:
import subprocess
import sys

# 既に導入済みなら不要な更新をせず、満たしていない依存だけを導入する。
subprocess.run([sys.executable, "-m", "pip", "install", *['numpy>=1.26,<3', 'torch>=2.6,<3', 'torchvision>=0.21,<1', 'mcp==2.3.0', 'PyYAML==6.0.3', 'opentelemetry-sdk==1.45.2', 'opentelemetry-exporter-otlp-proto-http==1.45.2']], check=True)


In [ ]:
from math import comb, sqrt


def mcnemar_exact(worse, better):
    """同一設問で判定が変わった対に対する、二項分布の両側exact検定。"""
    n = worse + better
    if n == 0:
        return 1.0
    tail = sum(comb(n, i) for i in range(min(worse, better) + 1)) / 2**n
    return min(1.0, 2 * tail)


def wilson(successes, n, z=1.96):
    p = successes / n
    den = 1 + z*z/n
    center = (p + z*z/(2*n)) / den
    half = z * sqrt(p*(1-p)/n + z*z/(4*n*n)) / den
    return center - half, center + half


# 仮定した確率の計算。実モデルの成功率ではない。
p, k = 0.6, 5
print(f"Pass@5 = {1 - (1-p)**k:.5f}, Pass^5 = {p**k:.5f}")
se = sqrt(0.7 * 0.3 / 100)
lo, hi = wilson(70, 100)
print(f"70/100: SE = {se:.5f}, 正規近似の半幅 = {1.96*se:.5f}")
print(f"Wilson 95%区間 = [{lo:.5f}, {hi:.5f}]")
for worse, better in [(0, 10), (20, 30)]:
    print(f"悪化{worse}・改善{better}: exact McNemar p = {mcnemar_exact(worse, better):.6f}")
